# 03 — Color
**Goal:** Choose colors with intention — palettes that are colorblind-safe, encode meaning, and look like they belong in a top-venue paper.

By the end of this notebook you will:
- Understand the three palette types: categorical, sequential, diverging
- Know which palettes are colorblind-safe and why it matters
- Be able to assign colors to encode meaning (not just aesthetics)
- See what color choices top ML labs actually use

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('research.mplstyle')   # load the style you built in notebook 02

np.random.seed(0)

---
## Part 1 — The three palette types

| Type | When to use | Example |
|------|------------|--------|
| **Categorical** | Distinguishing unordered groups (Model A vs B vs C) | Okabe-Ito, IBM |
| **Sequential** | Encoding magnitude (low → high), one variable | Blues, Viridis |
| **Diverging** | Encoding deviation from a midpoint (negative ↔ positive) | RdBu, coolwarm |

Choosing the wrong type is a common mistake — e.g., using a sequential palette for categorical groups makes readers think the order means something.

**Exercise:** Visualise all three types using matplotlib's built-in colormaps.

In [ ]:
import matplotlib.colors as mcolors

def show_palette(colors, title):
    """Display a list of colors as swatches."""
    fig, ax = plt.subplots(figsize=(len(colors) * 0.8, 0.8))
    for i, c in enumerate(colors):
        ax.add_patch(mpatches.Rectangle((i, 0), 1, 1, color=c))
    ax.set_xlim(0, len(colors))
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, fontsize=11)
    plt.show()

# TODO: use show_palette to display three palettes:
# 1. A categorical palette: 5 distinct colors of your choice
# 2. A sequential palette: 8 shades sampled from the 'Blues' colormap
#    Hint: plt.cm.Blues(np.linspace(0.2, 0.9, 8))
# 3. A diverging palette: 9 shades sampled from 'RdBu_r'


---
## Part 2 — Colorblind safety

~8% of men and ~0.5% of women have color vision deficiency. The most common type (red-green) makes standard red/green pairs indistinguishable. Reviewers and readers may be affected.

**The Okabe-Ito palette** was designed specifically to be distinguishable under all common forms of color blindness. It's used by many top ML papers.

```
Okabe-Ito colors:
#E69F00  orange
#56B4E9  sky blue
#009E73  green
#F0E442  yellow
#0072B2  blue
#D55E00  vermillion
#CC79A7  pink
#000000  black
```

**Exercise:** Display the Okabe-Ito palette using `show_palette`.

In [ ]:
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]

# TODO: display OKABE_ITO using show_palette


---
## Part 3 — Color to encode meaning in ML figures

Color should answer a question, not just look pretty. Common patterns in ML papers:

- **Your model vs. baselines:** use a vivid color for your method, muted grays for baselines
- **Train vs. val:** two contrasting colors from the same palette
- **Better vs. worse performance:** sequential or diverging encoding
- **Ablation variants:** same hue, different saturation/lightness

**Exercise:** Plot 4 models' training curves. Use Okabe-Ito colors but make "Our Model" pop (vivid) and the 3 baselines muted.

In [ ]:
epochs = np.arange(1, 51)

our_model  = 2.5 * np.exp(-0.10 * epochs) + 0.05 * np.random.randn(50)
baseline_1 = 2.5 * np.exp(-0.07 * epochs) + 0.12 + 0.06 * np.random.randn(50)
baseline_2 = 2.5 * np.exp(-0.06 * epochs) + 0.18 + 0.06 * np.random.randn(50)
baseline_3 = 2.5 * np.exp(-0.05 * epochs) + 0.22 + 0.06 * np.random.randn(50)

# TODO: plot all 4 curves
# - Our Model: vivid color (pick from OKABE_ITO), linewidth=2.2
# - Baselines: muted gray tones e.g. '#AAAAAA', '#888888', '#666666', linewidth=1.2
# - Label each line, add legend, axis labels


---
## Part 4 — Sequential color for a heatmap

Sequential colormaps work well for 2D grids where one variable has magnitude (e.g., attention weights, accuracy across hyperparameter sweeps).

**Rules:**
- Use perceptually uniform maps: `viridis`, `plasma`, `Blues`, `Oranges`
- Avoid `jet` / `rainbow` — they create false visual discontinuities
- Always add a colorbar

**Exercise:** Make a heatmap of a 6×6 "hyperparameter sweep" grid (simulated accuracy values).

In [ ]:
lr_values    = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
wd_values    = [0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
accuracy_grid = np.random.uniform(0.55, 0.92, size=(6, 6))
# Add a realistic pattern: peak near lr=1e-3, wd=1e-4
accuracy_grid[2, 2] = 0.94
accuracy_grid[1, 2] = 0.91
accuracy_grid[2, 1] = 0.89

# TODO: create an imshow heatmap using cmap='Blues'
# Add x-tick labels (wd_values), y-tick labels (lr_values)
# Add a colorbar with label 'Accuracy'
# Hint: ax.imshow(...), fig.colorbar(..., ax=ax)


---
## Part 5 — Diverging color for showing change

Diverging maps are ideal when you want to show values **relative to a baseline** — e.g., improvement over a baseline model per task.

**Rules:**
- Center the colormap at 0 (or the baseline value) using `vmin`/`vmax`
- Use `RdBu_r` or `coolwarm`

**Exercise:** Make a heatmap showing per-task improvement (%) of your model vs. a baseline, across 5 tasks × 4 metrics.

In [ ]:
tasks   = ['Task A', 'Task B', 'Task C', 'Task D', 'Task E']
metrics = ['Accuracy', 'F1', 'BLEU', 'ROUGE']
delta   = np.random.uniform(-8, 12, size=(5, 4))  # % improvement
delta[0, 0] = -6   # a couple of notable regressions
delta[3, 2] = -4

# TODO: imshow with cmap='RdBu_r', centered at 0 (vmin=-12, vmax=12)
# Add task labels (y-axis), metric labels (x-axis), colorbar
# Bonus: annotate each cell with the delta value using ax.text()


---
## Part 6 — Setting a custom color cycle

When you plot multiple lines without specifying colors, matplotlib cycles through its default color list. You can replace this cycle with Okabe-Ito so every plot is automatically colorblind-safe.

**Exercise:** Set the Okabe-Ito palette as the default color cycle, then plot 5 lines without specifying any colors.

In [ ]:
from matplotlib.cycler import cycler

# TODO: set the axes color cycle to OKABE_ITO using:
# plt.rcParams['axes.prop_cycle'] = cycler(color=OKABE_ITO)

# Then plot 5 lines without any color argument and verify the colors match Okabe-Ito
x = np.linspace(0, 4 * np.pi, 100)


---
## Summary

| Concept | What to remember |
|---------|------------------|
| Palette type | Categorical for groups, sequential for magnitude, diverging for deviation |
| Colorblind safety | Use Okabe-Ito for categorical; viridis/Blues for sequential |
| Avoid | `jet`, `rainbow`, arbitrary RGB colors |
| Your model | Vivid color; baselines in muted gray |
| Diverging maps | Always center at meaningful midpoint using `vmin`/`vmax` |
| Color cycle | Set `axes.prop_cycle` in your style sheet |

**Next:** `04_training_curves.ipynb` — the most common figure in ML papers: loss/accuracy curves with confidence bands and multi-run comparisons.